In [1]:
# Fine tuning

In [6]:
!pip3 install datasets tf-keras 'accelerate>=0.26.0' torch 


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 3.0 MB/s eta 0:00:002.5 MB/s eta 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
"""
Complete Fine-Tuning Script for Text Classification
Train DistilBERT on IMDB sentiment analysis with full tracking and visualization
"""

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import os

In [4]:

def check_gpu():
    """Verify GPU availability and display info."""
    print("="*70)
    print("SYSTEM CHECK")
    print("="*70)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("WARNING: No GPU detected. Training will be slower on CPU.")
    
    print("="*70 + "\n")

check_gpu()

SYSTEM CHECK
PyTorch version: 2.9.1+cu128
CUDA available: False



In [5]:

def load_and_explore_data(sample_size=None):
    """Load IMDB dataset and show statistics."""
    print("="*70)
    print("LOADING DATA")
    print("="*70)
    
    # Load full dataset
    dataset = load_dataset("imdb")
    
    print(f"Full dataset loaded:")
    print(f"  Training examples: {len(dataset['train']):,}")
    print(f"  Test examples: {len(dataset['test']):,}")
    
    # Create smaller subset if specified
    if sample_size:
        train_dataset = dataset['train'].shuffle(seed=42).select(range(sample_size))
        test_dataset = dataset['test'].shuffle(seed=42).select(range(sample_size // 5))
        print(f"\nUsing subset for faster training:")
        print(f"  Training: {len(train_dataset):,} examples")
        print(f"  Test: {len(test_dataset):,} examples")
    else:
        train_dataset = dataset['train']
        test_dataset = dataset['test']
    
    # Show examples
    print(f"\nExample reviews:")
    for i in range(2):
        label = "Positive" if train_dataset[i]['label'] == 1 else "Negative"
        print(f"\n  Example {i+1} ({label}):")
        print(f"  {train_dataset[i]['text'][:200]}...")
    
    # Statistics
    labels = [example['label'] for example in train_dataset]
    print(f"\nLabel distribution:")
    print(f"  Negative (0): {labels.count(0):,} ({labels.count(0)/len(labels)*100:.1f}%)")
    print(f"  Positive (1): {labels.count(1):,} ({labels.count(1)/len(labels)*100:.1f}%)")
    
    print("="*70 + "\n")
    
    return train_dataset, test_dataset

load_and_explore_data()

LOADING DATA
Full dataset loaded:
  Training examples: 25,000
  Test examples: 25,000

Example reviews:

  Example 1 (Negative):
  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...

  Example 2 (Negative):
  "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that ...

Label distribution:
  Negative (0): 12,500 (50.0%)
  Positive (1): 12,500 (50.0%)



(Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }))

In [7]:
def prepare_tokenizer_and_data(train_dataset, test_dataset, model_name="distilbert-base-uncased"):
    """Tokenize datasets."""
    print("="*70)
    print("TOKENIZATION")
    print("="*70)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"Tokenizer: {model_name}")
    print(f"Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"Max length: {tokenizer.model_max_length}")
    
    # Example tokenization
    example_text = train_dataset[0]['text'][:100]
    tokens = tokenizer.tokenize(example_text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    print(f"\nExample tokenization:")
    print(f"  Text: {example_text}...")
    print(f"  Tokens: {tokens[:10]}...")
    print(f"  Token IDs: {token_ids[:10]}...")
    print(f"  Number of tokens: {len(tokens)}")
    
    # Tokenize datasets
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=512
        )
    
    print(f"\nTokenizing datasets...")
    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)
    
    # Set format for PyTorch
    tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    
    print(f"Tokenization complete!")
    print("="*70 + "\n")
    
    return tokenizer, tokenized_train, tokenized_test


LOADING DATA
Full dataset loaded:
  Training examples: 25,000
  Test examples: 25,000

Example reviews:

  Example 1 (Negative):
  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...

  Example 2 (Negative):
  "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that ...

Label distribution:
  Negative (0): 12,500 (50.0%)
  Positive (1): 12,500 (50.0%)

TOKENIZATION
Tokenizer: distilbert-base-uncased
Vocabulary size: 30,522
Max length: 512

Example tokenization:
  Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...
  Tokens: ['i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video']...
  Token IDs: [1045, 12524, 1045, 2572,

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenization complete!



(DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
 	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 }
 ),
 Dataset({
     features: ['text', 'label', 'input_ids', 'attention_mask'],
     num_rows: 25000
 }),
 Da

In [8]:

def create_model(model_name="distilbert-base-uncased", num_labels=2):
    """Load pre-trained model and display architecture."""
    print("="*70)
    print("MODEL SETUP")
    print("="*70)
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Model: {model_name}")
    print(f"Device: {device}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB")
    
    print("="*70 + "\n")
    
    return model, device


MODEL SETUP


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: distilbert-base-uncased
Device: cpu
Total parameters: 66,955,010
Trainable parameters: 66,955,010
Model size: ~255.4 MB



In [9]:

def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, F1."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = (predictions == labels).mean()
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [10]:

def train_model(model, tokenized_train, tokenized_test, output_dir="./results"):
    """Train the model with progress tracking."""
    print("="*70)
    print("TRAINING")
    print("="*70)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        
        # Training parameters
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        
        # Optimization
        warmup_steps=100,
        weight_decay=0.01,
        
        # Evaluation and logging
        eval_strategy="steps",
        eval_steps=100,
        logging_dir='./logs',
        logging_steps=50,
        save_strategy="steps",
        save_steps=100,
        
        # Best model
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        
        # Performance
        fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
        
        seed=42
    )
    
    print(f"Training configuration:")
    print(f"  Epochs: {training_args.num_train_epochs}")
    print(f"  Batch size: {training_args.per_device_train_batch_size}")
    print(f"  Learning rate: {training_args.learning_rate}")
    print(f"  Mixed precision (fp16): {training_args.fp16}")
    
    total_steps = len(tokenized_train) // training_args.per_device_train_batch_size * training_args.num_train_epochs
    print(f"  Total training steps: {total_steps:,}")
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics
    )
    
    print(f"\nStarting training...")
    print("="*70)
    
    # Train
    train_result = trainer.train()
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
    print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")
    print("="*70 + "\n")
    
    return trainer


In [11]:

def evaluate_model(trainer, tokenizer, device):
    """Evaluate model and show detailed results."""
    print("="*70)
    print("EVALUATION")
    print("="*70)
    
    # Evaluate on test set
    eval_results = trainer.evaluate()
    
    print(f"Test Set Performance:")
    print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"  Precision: {eval_results['eval_precision']:.4f}")
    print(f"  Recall:    {eval_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {eval_results['eval_f1']:.4f}")
    print(f"  Loss:      {eval_results['eval_loss']:.4f}")
    
    # Test on custom examples
    print(f"\n" + "-"*70)
    print("Testing on custom examples:")
    print("-"*70)
    
    test_examples = [
        "This movie was absolutely amazing! Best film I've seen all year.",
        "Terrible movie. Complete waste of time and money.",
        "It was okay, nothing special but not terrible either.",
        "Incredible performances and stunning cinematography!",
        "Boring plot, bad acting, very disappointed."
    ]
    
    model = trainer.model
    model.eval()
    
    for text in test_examples:
        encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            probs = torch.softmax(outputs.logits, dim=1)
            prediction = torch.argmax(probs, dim=1)
        
        sentiment = "Positive ✓" if prediction.item() == 1 else "Negative ✗"
        confidence = probs[0][prediction.item()].item()
        
        print(f"\nText: {text}")
        print(f"Prediction: {sentiment} (confidence: {confidence:.1%})")
    
    print("="*70 + "\n")
    
    return eval_results


def create_visualizations(trainer, save_dir="./visualizations"):
    """Create and save training visualizations."""
    print("="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    
    os.makedirs(save_dir, exist_ok=True)
    
    # Get predictions
    predictions = trainer.predict(trainer.eval_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=-1)
    true_labels = predictions.label_ids
    
    # Confusion Matrix
    cm = confusion_matrix(true_labels, pred_labels)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title('Confusion Matrix - Sentiment Classification', fontsize=14, pad=20)
    plt.colorbar()
    
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ['Negative', 'Positive'])
    plt.yticks(tick_marks, ['Negative', 'Positive'])
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=16)
    
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    
    cm_path = os.path.join(save_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    print(f"✓ Confusion matrix saved to: {cm_path}")
    plt.close()
    
    # Prediction confidence distribution
    probs = torch.softmax(torch.tensor(predictions.predictions), dim=1)
    confidences = probs.max(dim=1).values.numpy()
    
    plt.figure(figsize=(10, 6))
    plt.hist(confidences, bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel('Prediction Confidence', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.title('Distribution of Prediction Confidences', fontsize=14, pad=20)
    plt.axvline(confidences.mean(), color='red', linestyle='--', 
                label=f'Mean: {confidences.mean():.3f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    conf_path = os.path.join(save_dir, 'confidence_distribution.png')
    plt.savefig(conf_path, dpi=300, bbox_inches='tight')
    print(f"✓ Confidence distribution saved to: {conf_path}")
    plt.close()
    
    print("="*70 + "\n")


def save_model(trainer, tokenizer, save_dir="./my_finetuned_model"):
    """Save the fine-tuned model and tokenizer."""
    print("="*70)
    print("SAVING MODEL")
    print("="*70)
    
    # Save model and tokenizer
    trainer.model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to: {save_dir}")
    print(f"Files saved:")
    for file in os.listdir(save_dir):
        print(f"  - {file}")
    
    print("\nTo load later:")
    print(f"  from transformers import AutoModelForSequenceClassification, AutoTokenizer")
    print(f"  model = AutoModelForSequenceClassification.from_pretrained('{save_dir}')")
    print(f"  tokenizer = AutoTokenizer.from_pretrained('{save_dir}')")
    
    print("="*70 + "\n")



In [13]:

def main(sample_size=1000):
    """
    Complete fine-tuning pipeline.
    
    Args:
        sample_size: Number of training examples to use (None for full dataset)
                    Recommended: 1000 for quick testing, None for best performance
    """
    print("\n" + "="*70)
    print("FINE-TUNING DISTILBERT FOR SENTIMENT ANALYSIS")
    print("="*70 + "\n")
    
    # Check system
    check_gpu()
    
    # Load data
    train_dataset, test_dataset = load_and_explore_data(sample_size=sample_size)
    
    # Tokenize
    tokenizer, tokenized_train, tokenized_test = prepare_tokenizer_and_data(
        train_dataset, test_dataset
    )
    
    # Create model
    model, device = create_model()
    
    # Train
    trainer = train_model(model, tokenized_train, tokenized_test)
    
    # Evaluate
    eval_results = evaluate_model(trainer, tokenizer, device)
    
    # Visualize
    create_visualizations(trainer)
    
    # Save
    save_model(trainer, tokenizer)
    
    print("="*70)
    print("ALL DONE!")
    print("="*70)
    print(f"\nFinal Accuracy: {eval_results['eval_accuracy']:.2%}")
    print("\nNext steps:")
    print("  1. Try the full dataset: main(sample_size=None)")
    print("  2. Experiment with different hyperparameters")
    print("  3. Try a different dataset (AG News, SST-2)")
    print("  4. Move on to Project 3: Attention mechanism")
    print("="*70 + "\n")


if __name__ == "__main__":
    # Run with small sample for quick testing
    # Change to main(sample_size=None) to use full dataset
    main(sample_size=300)


FINE-TUNING DISTILBERT FOR SENTIMENT ANALYSIS

SYSTEM CHECK
PyTorch version: 2.9.1+cu128
CUDA available: False

LOADING DATA
Full dataset loaded:
  Training examples: 25,000
  Test examples: 25,000

Using subset for faster training:
  Training: 300 examples
  Test: 60 examples

Example reviews:

  Example 1 (Positive):
  There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...

  Example 2 (Positive):
  This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub y...

Label distribution:
  Negative (0): 150 (50.0%)
  Positive (1): 150 (50.0%)

TOKENIZATION
Tokenizer: distilbert-base-uncased
Vocabulary size: 30,522
Max length: 512

Example tokenization:
  Text: There is no relation at all between Fortier 

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenization complete!

MODEL SETUP


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...


TRAINING COMPLETE
Training time: 415.88 seconds
Training samples/second: 2.16

EVALUATION


/home/dat/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test Set Performance:
  Accuracy:  0.6667
  Precision: 0.7273
  Recall:    0.5333
  F1 Score:  0.6154
  Loss:      0.6726

----------------------------------------------------------------------
Testing on custom examples:
----------------------------------------------------------------------

Text: This movie was absolutely amazing! Best film I've seen all year.
Prediction: Positive ✓ (confidence: 50.5%)

Text: Terrible movie. Complete waste of time and money.
Prediction: Negative ✗ (confidence: 53.1%)

Text: It was okay, nothing special but not terrible either.
Prediction: Positive ✓ (confidence: 51.9%)

Text: Incredible performances and stunning cinematography!
Prediction: Positive ✓ (confidence: 52.5%)

Text: Boring plot, bad acting, very disappointed.
Prediction: Negative ✗ (confidence: 52.5%)

CREATING VISUALIZATIONS


/home/dat/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✓ Confusion matrix saved to: ./visualizations/confusion_matrix.png
✓ Confidence distribution saved to: ./visualizations/confidence_distribution.png

SAVING MODEL
Model and tokenizer saved to: ./my_finetuned_model
Files saved:
  - config.json
  - tokenizer_config.json
  - vocab.txt
  - model.safetensors
  - tokenizer.json
  - special_tokens_map.json

To load later:
  from transformers import AutoModelForSequenceClassification, AutoTokenizer
  model = AutoModelForSequenceClassification.from_pretrained('./my_finetuned_model')
  tokenizer = AutoTokenizer.from_pretrained('./my_finetuned_model')

ALL DONE!

Final Accuracy: 66.67%

Next steps:
  1. Try the full dataset: main(sample_size=None)
  2. Experiment with different hyperparameters
  3. Try a different dataset (AG News, SST-2)
  4. Move on to Project 3: Attention mechanism

